In [1]:
import os
import cv2
import json
import math
import shutil
import copy
from pathlib import Path
import albumentations as A

# =========================================================
# 1. ĐƯỜNG DẪN 
# =========================================================
IN_TRAIN_DIR = Path("/kaggle/input/datasets/quii29/rukopys-dataset/train")
IN_IMG_DIR = IN_TRAIN_DIR / "images"
IN_META_PATH = IN_TRAIN_DIR / "metadata.jsonl"

OUT_ROOT = Path("/kaggle/working/rukopys_augmented")
OUT_TRAIN_DIR = OUT_ROOT / "train"
OUT_IMG_DIR = OUT_TRAIN_DIR / "images"
OUT_META_PATH = OUT_TRAIN_DIR / "metadata.jsonl"

OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 2. CẤU HÌNH AUGMENTATION
# =========================================================
RARE_CLASSES = ["formula", "table", "annotation", "image", "graph"]
SRC_WEIGHTS = {"archive": 1.5, "dictation": 1.0, "school": 1.0, "university": 1.0}

aug_dictation = A.Compose([
    A.GaussNoise(p=0.7), 
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.7),
    A.Blur(blur_limit=5, p=0.5),
    A.ISONoise(p=0.5)
], bbox_params=A.BboxParams(format='coco', label_fields=['region_idx']))

aug_archive = A.Compose([
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-7, 7), p=0.7),
    A.Perspective(scale=(0.01, 0.05), p=0.5)
], bbox_params=A.BboxParams(format='coco', label_fields=['region_idx'], min_visibility=0.3))

# =========================================================
# 3. XỬ LÝ DỮ LIỆU
# =========================================================
print("🚀 Bắt đầu xử lý...")
count_orig = 0
count_aug = 0
error_logs = []

with open(IN_META_PATH, 'r', encoding='utf-8') as f_in, \
     open(OUT_META_PATH, 'w', encoding='utf-8') as f_out:
    
    for line in f_in:
        data = json.loads(line)
        fname = Path(data["file_name"]).name 
        orig_img_path = IN_IMG_DIR / fname
        
        if not orig_img_path.exists(): 
            continue
            
        shutil.copy2(orig_img_path, OUT_IMG_DIR / fname)
        data["file_name"] = f"images/{fname}"
        f_out.write(json.dumps(data, ensure_ascii=False) + "\n")
        count_orig += 1
        
        regions = data.get("regions", [])
        rare_count = sum(1 for r in regions if r["type"] in RARE_CLASSES)
        src = data.get("source", "dictation")
        
        base_w = SRC_WEIGHTS.get(src, 1.0)
        rare_factor = 1.0 + (1.0 * rare_count if rare_count > 0 else 0.0)
        repeats = min(int(math.ceil(base_w * rare_factor)), 5)
        
        if repeats > 1 and regions:
            image = cv2.imread(str(orig_img_path))
            if image is None:
                continue
                
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            h_img, w_img = image.shape[:2]
            
            valid_bboxes = []
            valid_idx = []
            
            for idx, r in enumerate(regions):
                x, y, w_box, h_box = r["bbox"]
                
                x_safe = max(0.0, float(x))
                y_safe = max(0.0, float(y))
                w_safe = max(1.0, min(float(w_box), w_img - x_safe))
                h_safe = max(1.0, min(float(h_box), h_img - y_safe))
                
                if w_safe > 1.0 and h_safe > 1.0:
                    valid_bboxes.append([x_safe, y_safe, w_safe, h_safe])
                    valid_idx.append(idx)
            
            if not valid_bboxes:
                continue
                
            for i in range(1, repeats):
                try:
                    transform = aug_dictation if src == 'dictation' else aug_archive
                    transformed = transform(image=image, bboxes=valid_bboxes, region_idx=valid_idx)
                    trans_img = transformed['image']
                    trans_bboxes = transformed['bboxes']
                    trans_idx = transformed['region_idx']
                    
                    if not trans_bboxes: continue 
                    
                    new_fname = f"{Path(fname).stem}_aug{i}{Path(fname).suffix}"
                    trans_img = cv2.cvtColor(trans_img, cv2.COLOR_RGB2BGR)
                    cv2.imwrite(str(OUT_IMG_DIR / new_fname), trans_img)
                    
                    new_data = copy.deepcopy(data)
                    new_data["file_name"] = f"images/{new_fname}"
                    
                    new_regions = []
                    for bbox, idx in zip(trans_bboxes, trans_idx):
                        r = copy.deepcopy(data["regions"][int(idx)])
                        r["bbox"] = [round(v, 2) for v in bbox]
                        new_regions.append(r)
                        
                    new_data["regions"] = new_regions
                    f_out.write(json.dumps(new_data, ensure_ascii=False) + "\n")
                    count_aug += 1
                    
                except Exception as e:
                    if len(error_logs) < 5:
                        error_logs.append(f"Lỗi Aug tại {fname}: {str(e)}")

print(f"\n✅ Xử lý xong! Gốc: {count_orig} | Sinh thêm: {count_aug}")

if error_logs:
    print("\n⚠️ Một số lỗi đã gặp (Đã tự động bỏ qua an toàn):")
    for log in error_logs:
        print(f" - {log}")

# =========================================================
# 4. NÉN ZIP VÀ DỌN DẸP
# =========================================================
print("\n📦 Đang nén thành file ZIP...")
%cd /kaggle/working
!zip -rq rukopys_augmented_original_format.zip rukopys_augmented/

print("🧹 Đang dọn dẹp các tập tin tạm...")
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)

print("🎉 ĐÃ XONG! File 'rukopys_augmented_original_format.zip' đã sẵn sàng và ổ cứng đã được dọn sạch.")

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


🚀 Bắt đầu xử lý...


Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG



✅ Xử lý xong! Gốc: 1330 | Sinh thêm: 1766

📦 Đang nén thành file ZIP...
/kaggle/working
🧹 Đang dọn dẹp các tập tin tạm...
🎉 ĐÃ XONG! File 'rukopys_augmented_original_format.zip' đã sẵn sàng và ổ cứng đã được dọn sạch.
